In [5]:
import subprocess
import json
import requests
from datetime import datetime, timedelta
from urllib.parse import quote

# Execute the curl command to retrieve the access token
response = subprocess.run(['curl', '-H', 'Metadata-Flavor: Google', 'http://169.254.169.254/computeMetadata/v1/instance/service-accounts/default/token'], capture_output=True, text=True)

# Check if the command was successful
if response.returncode == 0:
    # Extract the access token from the response
    token = json.loads(response.stdout)['access_token']
else:
    # Print an error message if the command failed
    print("Error:", response.stderr)

# Define the project ID
project_id = "fdc-development-408605"

# Define the current time and interval
end_time = datetime.utcnow()
start_time = end_time - timedelta(hours=3)

# Define filter string
filter_str = 'resource.type="k8s_container" AND resource.label."pod_name"="jy-4fa3fabe-5506-4a55-a228-1f7238b62dd3-sunny10683796ltimindt" AND metric.type="kubernetes.io/container/memory/limit_utilization"'
encoded_filter_str = quote(filter_str, safe='')

# Construct the API URL
sA = "ALIGN_MEAN"
sR = "REDUCE_MEAN"
url = f'https://monitoring.googleapis.com/v3/projects/{project_id}/timeSeries?filter={encoded_filter_str}&aggregation.perSeriesAligner={sA}&aggregation.crossSeriesReducer={sR}&aggregation.alignmentPeriod=+1800s&interval.startTime={start_time.strftime("%Y-%m-%dT%H:%M:%SZ")}&interval.endTime={end_time.strftime("%Y-%m-%dT%H:%M:%SZ")}'

# Make the API call
response = requests.get(
    url,
    headers={"Authorization": f"Bearer {token}"}
)

# Parse the response and fetch the first doubleValue
first_double_value = response.json()['timeSeries'][0]['points'][0]['value']['doubleValue']

# Print the first doubleValue
print("First doubleValue:", first_double_value)
print('response json')
print(response.json())

first_point = response.json()['timeSeries'][0]['points'][0]
start_time = first_point['interval']['startTime']
double_value = first_point['value']['doubleValue']

print('end_tome', end_time)
print("time:", start_time)
print("Cpu utilisatin:", double_value)

First doubleValue: 0.010917450496128625
response json
{'timeSeries': [{'metric': {'type': 'kubernetes.io/container/memory/limit_utilization'}, 'resource': {'type': 'k8s_container', 'labels': {'project_id': 'fdc-development-408605'}}, 'metricKind': 'GAUGE', 'valueType': 'DOUBLE', 'points': [{'interval': {'startTime': '2024-03-28T14:45:24Z', 'endTime': '2024-03-28T14:45:24Z'}, 'value': {'doubleValue': 0.010917450496128625}}, {'interval': {'startTime': '2024-03-28T14:15:24Z', 'endTime': '2024-03-28T14:15:24Z'}, 'value': {'doubleValue': 0.010888713110060916}}, {'interval': {'startTime': '2024-03-28T13:45:24Z', 'endTime': '2024-03-28T13:45:24Z'}, 'value': {'doubleValue': 0.009585927327473959}}]}], 'unit': '1'}
end_tome 2024-03-28 14:45:24.306905
time: 2024-03-28T14:45:24Z
Cpu utilisatin: 0.010917450496128625
